In [4]:
import sys
from pathlib import Path
import traceback

# Path injection for Notebook testing
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from torch import long
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.optim.lr_scheduler import ReduceLROnPlateau

from dataset import AsTensor 
from gpt.dataset import TinyShakes
from learning import Learn, Metric, Selector
from model import GPT

logger = Metric.setup_logging(log_name='train_job', log_dir='./data')

def run_training():
    dir = "./data"

    d_seq = 25       # dimension sequence length
    d_gen = 25       # dimension generate number of tokens
    d_vocab = 50304  # dimension vocabulary
    d_vec = 384      # dimension embedding vector
    d_model = 384    # dimension model input

    assert d_model == d_vec

    model_param = {
        'd_model': d_model,
        'd_vocab': d_vocab, 
        'n_head': 6, 
        'num_layers': 6,
        'd_gen': d_gen,
        'd_vec': d_vec,
        'temperature': 1000,
        'top_k': 3,
        'embed_param': {
            'tokens': (d_vocab, d_vec, None, True), 
            'y': (d_vocab, d_vec, None, True),
            'position': (d_seq, d_vec, None, True)
        }
    } 
    
    ds_param = {
        'train_param': {
            'transforms': {
                'tokens': [AsTensor(long)],
                'y': [AsTensor(long)],
                'position': [AsTensor(long)]
            },
            'n': 500,
            'd_seq': d_seq,
            'dir': dir,
            'prompt': False,
        }
    }

    metric_param = {
        'metric_name': 'transformer',
        'report_interval': 1,
        'last_n': 10,
        'min_lr': 0.0025  # break if learning rate falls below
    } 
                
    opt_param = {'lr': 0.01}
    crit_param = {}
    sample_param = {'set_seed': 88, 'splits': (0.7, 0.15)}
    sched_param = {'factor': 0.5, 'patience': 2, 'cooldown': 2}
    
    learner = Learn(
        [TinyShakes], GPT, Metric=Metric, Sampler=Selector, 
        Optimizer=Adam, Scheduler=ReduceLROnPlateau, Criterion=CrossEntropyLoss,
        model_param=model_param, ds_param=ds_param, metric_param=metric_param,
        opt_param=opt_param, crit_param=crit_param, sample_param=sample_param, 
        sched_param=sched_param,
        dir=dir, batch_size=32, epoch=3, gpu=False,
        save_model='tinyshakes384', load_model='tinyshakes384'
    )
    
    logger.info("🚀 train_job initialized..")

    try:
        out = learner.run_experiment()
        logger.info("✅ train_job complete... {}".format(out))
    except Exception as e:
        full_trace = traceback.format_exc()
        logger.error(f"main.handle_text failed: {e}\n{full_trace}")
        raise Exception({"message": str(e), "traceback": full_trace})

if __name__ == "__main__":
    run_training()

2026-04-19 10:12:36,338 [train_job] INFO: train_job logging initialized at: ./data/train_job_20260419_101236.log
2026-04-19 10:12:36,340 [gpt.dataset] INFO: TinyShakes.load_data inyshakes.txt loaded from saved file in ./data
2026-04-19 10:12:36,341 [gpt.dataset] INFO: TinyShakes.load_data tokens loaded from file ./data/tinyshakes_stripped_encoded.bin
2026-04-19 10:12:36,346 [cosmosis.dataset] INFO: CDataset.__init__ created...
2026-04-19 10:12:36,710 [model] INFO: CModel.init_weights applying _init_weights...
2026-04-19 10:12:36,801 [model] INFO: CModel.__init__GPT model loaded...
2026-04-19 10:12:36,802 [model] INFO: CModel.get_num_params number of model parameters: 19317120
2026-04-19 10:12:36,860 [learning] INFO: learn.__init__ model loaded from: tinyshakes384.pth
2026-04-19 10:12:37,001 [learning] INFO: learn.__init__ embedding weights loaded successfully...
2026-04-19 10:12:37,002 [learning] INFO: learn.__init__ model loaded: GPT
2026-04-19 10:12:37,010 [learning] INFO: learn.__in

In [2]:
%tb

SystemExit: 1

In [5]:
import sys
from pathlib import Path

# Path injection for Notebook testing
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from torch import long

from gpt.dataset import TinyShakes
from cosmosis.learning import Learn, Metric, Selector
from cosmosis.model import GPT
from cosmosis.dataset import AsTensor

logger = Metric.setup_logging(log_name='backend.main', log_dir='./data')

dir = "./data"

d_seq = 25       # dimension sequence length
d_gen = 50       # dimension generate number of tokens
d_vocab = 50304  # dimension vocabulary
d_vec = 384      # dimension embedding vector
d_model = 384    # dimension model input

assert d_model == d_vec

ds_param = {'train_param': {'transforms': {'tokens': [AsTensor(long)],
                                            'y': [AsTensor(long)],
                                            'position': [AsTensor(long)]},
                            'n': 1, # set to 1 for inference
                            'dir': dir,
                            'prompt': None},
            }

model_param = {'d_model': d_model,
                'd_vocab': d_vocab, 
                'n_head': 6, 
                'num_layers': 6,
                'd_gen': d_gen,
                'd_vec': d_vec,
                'temperature': 1000,
                'top_k': 3,
                'embed_param': {'tokens': (d_vocab, d_vec, None, True), 
                                #'y': (d_vocab, d_vec, None, True),
                                'position': (d_gen, d_vec, None, True)},
                } 
                                    
metric_param = {'metric_name': 'transformer'}                        
            
opt_param = {}
crit_param = {}
sample_param = {}
sched_param = {}

learner = Learn(
    [TinyShakes], GPT, Metric=Metric, Sampler=Selector, 
    Optimizer=None, Scheduler=None, Criterion=None,
    model_param=model_param, ds_param=ds_param, metric_param=metric_param,
    opt_param=opt_param, crit_param=crit_param, sample_param=sample_param, 
    sched_param=sched_param, batch_size=1, epoch=1,
    dir=dir, save_model=False, load_model='tinyshakes384', 
    gpu=False)


text_input = "apple orange banana"

learner.run_experiment(prompt=text_input)


2026-04-19 10:13:45,340 [backend.main] INFO: backend.main logging initialized at: ./data/backend.main_20260419_101345.log
2026-04-19 10:13:45,344 [cosmosis.dataset] INFO: CDataset.__init__ created...
2026-04-19 10:13:45,612 [cosmosis.model] INFO: CModel.init_weights applying _init_weights...
2026-04-19 10:13:45,697 [cosmosis.model] INFO: CModel.__init__GPT model loaded...
2026-04-19 10:13:45,699 [cosmosis.model] INFO: CModel.get_num_params number of model parameters: 19317120
2026-04-19 10:13:45,758 [cosmosis.learning] INFO: learn.__init__ model loaded from: tinyshakes384.pth
2026-04-19 10:13:45,782 [cosmosis.learning] INFO: learn.__init__ embedding weights loaded successfully...
2026-04-19 10:13:45,783 [cosmosis.learning] INFO: learn.__init__ model loaded: GPT
2026-04-19 10:13:45,792 [cosmosis.learning] INFO: learn.__init__ running model on cpu...
2026-04-19 10:13:45,793 [cosmosis.learning] INFO: learn.__init__ inference engine ready...
2026-04-19 10:13:45,809 [cosmosis.model] INFO: G

{'learn.infer predictions': array([['ranking Cars CAP CAP Fredape PCB animated intense TraffordIAiza Fr outpostecttechnpse Empiremiddle solitaryRen\\" Zi boxing wentpectingGordon cited interventioncadeNationpsearantine SMSフォ Country newsprefRiver Jal Ireligible approx grilled Hopefully diffusionoooo Slowlyirming inspector']],
       dtype='<U285')}

In [1]:
import sys
from pathlib import Path

# Path injection for Notebook testing
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from torch import long

from torch.utils.data import Sampler, DataLoader

from gpt.dataset import TinyShakes
from cosmosis.learning import Learn, Metric, Selector
from cosmosis.model import GPT
from cosmosis.dataset import AsTensor

logger = Metric.setup_logging(log_name='backend.main', log_dir='./data')



dir = "./data"
d_gen = 25 # dimension generate number of tokens
d_vocab = 50304 # dimension vocabulary
d_vec = 384 # dimension embedding vector
d_model = 384 # dimension model input
d_pos = 25 # dimension positional encoding d_pos >= max(len(prompt_tokens), d_gen)

assert d_model == d_vec

ds_param = {'train_param': {'transforms': {'tokens': [AsTensor(long)],
                                            'y': [AsTensor(long)],
                                            'position': [AsTensor(long)]},
                            'n': 1, # set to 1 for inference
                            'dir': dir,
                            'prompt': None},
            }

model_param = {'d_model': d_model,
                'd_vocab': d_vocab, 
                'n_head': 6, 
                'num_layers': 6,
                'd_gen': d_gen,
                'd_vec': d_vec,
                'temperature': 1000,
                'top_k': 3,
                'embed_param': {'tokens': (d_vocab, d_vec, None, True), 
                                #'y': (d_vocab, d_vec, None, True),
                                'position': (d_pos, d_vec, None, True)},
                } 
                                    
metric_param = {'metric_name': 'transformer'}                        
            
opt_param = {}
crit_param = {}
sample_param = {}
sched_param = {}

dataset = [10, 20, 30]

sampler = Selector(dataset, **sample_param)

print(list(sampler('infer'))) # should print [10, 20, 30]

dataloader = DataLoader(dataset, batch_size=1, 
                                sampler=sampler('infer'), 
                                num_workers=1, 
                                pin_memory=False, 
                                drop_last=False)


for d in dataloader:
    print(d)

2026-04-19 10:06:48,620 [backend.main] INFO: backend.main logging initialized at: ./data/backend.main_20260419_100648.log
[20, 10, 30]


IndexError: Caught IndexError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/fltr/anaconda3/envs/sagan-gpt/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/fltr/anaconda3/envs/sagan-gpt/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
IndexError: list index out of range
